In [1]:
# ============================================================================
# PREREQUISITE SETUP - RUN THIS FIRST
# ============================================================================

# Install libraries
!pip install xgboost lightgbm catboost shap scikit-learn matplotlib seaborn plotly -q

# libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# ML Libraries
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.metrics import (classification_report, accuracy_score, precision_score,
                              recall_score, f1_score, confusion_matrix, roc_auc_score,
                              roc_curve, precision_recall_curve)
from sklearn.preprocessing import label_binarize
import joblib
import pickle
import json
from datetime import datetime
import time

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# ============================================================================
# DATA LOADING AND PREPARATION
# ============================================================================

# Load your dataset (update the path)
df = pd.read_csv('/content/drive/MyDrive/AIvy - AI Powered Job Recommendation System/Pace 02 - model training /Data set/processed data set new.csv')  # UPDATE THIS PATH

# ============================================================================
# MODIFICATION START: Sample 1000 rows
# ============================================================================
TARGET_ROWS = 1000
if len(df) > TARGET_ROWS:

    try:
        df_sampled = df.groupby('job_title', group_keys=False).apply(
            lambda x: x.sample(int(np.rint(TARGET_ROWS * len(x) / len(df))), random_state=42)
        )
        # Ensure we don't exceed TARGET_ROWS due to rounding
        df_sampled = df_sampled.head(TARGET_ROWS)

        # If stratified sampling didn't get exactly 1000 rows or failed, fall back to simple sample
        if len(df_sampled) != TARGET_ROWS:
            df_sampled = df.sample(n=TARGET_ROWS, random_state=42).reset_index(drop=True)
            print(f" Fell back to simple random sampling for 1000 rows due to class imbalance.")

        df = df_sampled
    except Exception:
        df = df.sample(n=TARGET_ROWS, random_state=42).reset_index(drop=True)
        print(f"⚠️ Simple random sampling used for 1000 rows.")

print("\n" + "="*70)
print(f"DATASET OVERVIEW (REDUCED TO {len(df)} ROWS)")
print("="*70)
# ============================================================================
# MODIFICATION END
# ============================================================================

# Display basic info
print(f"Shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst few rows:")
print(df.head())

# Define features and target
feature_cols = [
    'search_country_encoded', 'job_level_encoded', 'job_type_onsite',
    'job_title_word_count_scaled', 'skills_word_count_scaled',
    'job_title_char_count_scaled', 'skills_char_count_scaled',
    'skills_list_count_scaled', 'title_tokens_count_scaled',
    'skills_tokens_count_scaled', 'job_level_encoded_scaled',
    'search_country_encoded_scaled',
    'has_skill_communication', 'has_skill_customer_service',
    'has_skill_teamwork', 'has_skill_leadership',
    'has_skill_communication_skills'
] + [f'tfidf_component_{i}' for i in range(20)]

# Prepare X and y
X = df[feature_cols].fillna(0)
y = df['job_title']

# 2. Handle rare classes
class_counts = y.value_counts()

y = y.apply(lambda x: x if class_counts[x] >= 2 else "Other")

# 3. Train-test split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\n Features: {len(feature_cols)}")
print(f" Samples: {len(X)}") # Should be 1000
print(f"\n Target Distribution (after rare class handling):")
print(y.value_counts())


print(f"\n Training samples: {len(X_train)}") # Should be 800 (80% of 1000)
print(f" Test samples: {len(X_test)}")      # Should be 200 (20% of 1000)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 7.3 MB/s eta 0:00:00
Mounted at /content/drive
 Fell back to simple random sampling for 1000 rows due to class imbalance.

DATASET OVERVIEW (REDUCED TO 1000 ROWS)
Shape: (1000, 66)

Columns: ['job_link', 'last_processed_time', 'got_summary', 'got_ner', 'is_being_worked', 'job_title', 'company', 'job_location', 'first_seen', 'search_city', 'search_country', 'search_position', 'job_level', 'job_type', 'job_skills', 'job_title_clean', 'company_clean', 'job_skills_clean', 'job_location_clean', 'job_title_word_count', 'skills_word_count', 'job_title_char_count', 'skills_char_count', 'skills_list_count', 'has_skill_communication', 'has_skill_customer_service', 'has_skill_teamwork', 'has_skill_leadership', 'has_skill_communication_skills', 'job_title_tokens', 'job_skills_tokens', 'title_tokens_count', 'skills_tokens_count', 'combined_text_clean', 'job_level_encoded', 'job_type_onsite', 'search_country_encoded', 'job_title_word_count_sca

In [2]:
# ============================================================================
# MODEL 6: LOGISTIC REGRESSION
# ============================================================================

print("TRAINING LOGISTIC REGRESSION...")

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# Create model directory
model_dir = f'/content/drive/MyDrive/AIvy - AI Powered Job Recommendation System/Pace 02 - model training /Notebooks/Model_06_Logistic_Regression.ipynb'

TRAINING LOGISTIC REGRESSION...


In [3]:
# ============================================================================
# PART 1: EDA BEFORE TRAINING
# ============================================================================

print("\nPART 1: PRE-TRAINING EDA")

# 1. Feature Scaling for Logistic Regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Save scaler
joblib.dump(scaler, f'/content/drive/MyDrive/AIvy - AI Powered Job Recommendation System/Results/Model 06 - outputs-scaler.pkl')

# Plot scaled vs original features
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
sample_features = ['job_title_word_count_scaled', 'skills_word_count_scaled',
                   'tfidf_component_0', 'tfidf_component_1']

for idx, feat in enumerate(sample_features):
    row, col = idx // 2, idx % 2
    feat_idx = feature_cols.index(feat)

    # Original
    axes[row, col].hist(X_train[feat], bins=50, alpha=0.5, label='Original', color='blue', edgecolor='black')
    # Scaled
    axes[row, col].hist(X_train_scaled[:, feat_idx], bins=50, alpha=0.5, label='Scaled', color='yellow', edgecolor='black')
    axes[row, col].set_title(f'{feat}', fontsize=10, fontweight='bold')
    axes[row, col].set_xlabel('Value')
    axes[row, col].set_ylabel('Frequency')
    axes[row, col].legend()

plt.suptitle('Original vs Scaled Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'/content/drive/MyDrive/AIvy - AI Powered Job Recommendation System/Results/Model 06 - outputs-01_feature_scaling.png', dpi=300, bbox_inches='tight')
plt.close()

# 2. Feature Variance Analysis
feature_variance = np.var(X_train_scaled, axis=0)
variance_df = pd.DataFrame({
    'feature': feature_cols,
    'variance': feature_variance
}).sort_values('variance', ascending=False).head(20)

plt.figure(figsize=(12, 8))
plt.barh(variance_df['feature'], variance_df['variance'], color='gold', edgecolor='black')
plt.xlabel('Variance', fontsize=12, fontweight='bold')
plt.ylabel('Feature', fontsize=12, fontweight='bold')
plt.title('Top 20 Features by Variance (Scaled)', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig(f'/content/drive/MyDrive/AIvy - AI Powered Job Recommendation System/Results/Model 06 - outputs-02_feature_variance.png', dpi=300, bbox_inches='tight')
plt.close()

variance_df.to_csv(f'/content/drive/MyDrive/AIvy - AI Powered Job Recommendation System/Results/Model 06 - outputs-feature_variance.csv', index=False)

# 3. Multicollinearity Check (VIF - Variance Inflation Factor)
from sklearn.linear_model import LinearRegression

# Calculate VIF for top 15 features (to avoid computational overhead)
vif_features = feature_cols[:15]
vif_data = []

for i, feat in enumerate(vif_features):
    X_temp = X_train_scaled[:, [feature_cols.index(f) for f in vif_features if f != feat]]
    y_temp = X_train_scaled[:, feature_cols.index(feat)]

    lr = LinearRegression()
    lr.fit(X_temp, y_temp)
    r_squared = lr.score(X_temp, y_temp)

    vif = 1 / (1 - r_squared) if r_squared < 0.99 else 999
    vif_data.append({'feature': feat, 'VIF': vif})

vif_df = pd.DataFrame(vif_data).sort_values('VIF', ascending=False)

plt.figure(figsize=(12, 8))
colors = ['red' if vif > 10 else 'orange' if vif > 5 else 'green' for vif in vif_df['VIF']]
plt.barh(vif_df['feature'], vif_df['VIF'], color=colors, edgecolor='black')
plt.axvline(x=10, color='red', linestyle='--', linewidth=2, label='High Multicollinearity (VIF>10)')
plt.axvline(x=5, color='orange', linestyle='--', linewidth=2, label='Moderate (VIF>5)')
plt.xlabel('VIF Value', fontsize=12, fontweight='bold')
plt.ylabel('Feature', fontsize=12, fontweight='bold')
plt.title('Variance Inflation Factor (VIF) - Multicollinearity Check', fontsize=14, fontweight='bold')
plt.legend()
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig(f'/content/drive/MyDrive/AIvy - AI Powered Job Recommendation System/Results/Model 06 - outputs-03_multicollinearity_vif.png', dpi=300, bbox_inches='tight')
plt.close()

vif_df.to_csv(f'/content/drive/MyDrive/AIvy - AI Powered Job Recommendation System/Results/Model 06 - outputs-vif_analysis.csv', index=False)

# 4. Class Separability Visualization (2D PCA)
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_train_scaled)

plt.figure(figsize=(12, 8))
for class_label in np.unique(y_train):
    mask = y_train == class_label
    plt.scatter(X_pca[mask, 0], X_pca[mask, 1], label=f'Class {class_label}', alpha=0.6, s=50)

plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.2f}% variance)', fontsize=12, fontweight='bold')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.2f}% variance)', fontsize=12, fontweight='bold')
plt.title('Class Separability (PCA 2D Projection)', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'/content/drive/MyDrive/AIvy - AI Powered Job Recommendation System/Results/Model 06 - outputs04_class_separability_pca.png', dpi=300, bbox_inches='tight')
plt.close()

print("Pre-training EDA visualizations saved!")


PART 1: PRE-TRAINING EDA
Pre-training EDA visualizations saved!


In [4]:
# ============================================================================
# PART 2: HYPERPARAMETER TUNING (3 ITERATIONS)
# ============================================================================

print("\n PART 2: HYPERPARAMETER TUNING (3 Iterations)")

param_grids = [
    # Iteration 1: Broad search - Different regularization types
    {
        'C': [0.01, 0.1, 1, 10],
        'penalty': ['l1', 'l2'],
        'solver': ['saga'],
        'max_iter': [1000]
    },
    # Iteration 2: Refined search - Focus on best penalty
    {
        'C': [0.05, 0.1, 0.5, 1, 5],
        'penalty': ['l2'],
        'solver': ['lbfgs', 'saga'],
        'max_iter': [1000, 2000]
    },
    # Iteration 3: Fine-tuning - Optimal range
    {
        'C': [0.1, 0.3, 0.5, 0.7, 1.0],
        'penalty': ['l2'],
        'solver': ['lbfgs'],
        'max_iter': [2000],
        'class_weight': [None, 'balanced']
    }
]

tuning_results = []

for iteration, param_grid in enumerate(param_grids, 1):
    print(f"\n{'='*60}")
    print(f"Iteration {iteration}/3")
    print(f"{'='*60}")

    base_model = LogisticRegression(
        random_state=42,
        n_jobs=-1
    )

    grid_search = GridSearchCV(
        estimator=base_model,
        param_grid=param_grid,
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
        scoring='f1_weighted',
        n_jobs=-1,
        verbose=2
    )

    start_time = time.time()
    grid_search.fit(X_train_scaled, y_train)
    end_time = time.time()

    tuning_results.append({
        'iteration': iteration,
        'best_params': grid_search.best_params_,
        'best_score': grid_search.best_score_,
        'time_taken': end_time - start_time
    })

    print(f"\nBest Parameters: {grid_search.best_params_}")
    print(f"Best CV Score: {grid_search.best_score_:.4f}")
    print(f"Time Taken: {end_time - start_time:.2f}s")

# Save tuning results
with open(f'/content/drive/MyDrive/AIvy - AI Powered Job Recommendation System/Results/Model 06 - outputs-hyperparameter_tuning_results.json', 'w') as f:
    json.dump(tuning_results, f, indent=4)

best_iteration = max(tuning_results, key=lambda x: x['best_score'])
best_params = best_iteration['best_params']

print(f"\nBest Overall Parameters (Iteration {best_iteration['iteration']}):")
print(json.dumps(best_params, indent=4))

# Visualize hyperparameter tuning progression
tuning_scores = [result['best_score'] for result in tuning_results]
plt.figure(figsize=(10, 6))
plt.plot(range(1, 4), tuning_scores, marker='o', linestyle='-', linewidth=2, markersize=12, color='gold')
plt.xlabel('Tuning Iteration', fontsize=12, fontweight='bold')
plt.ylabel('Best F1 Score', fontsize=12, fontweight='bold')
plt.title('Hyperparameter Tuning Progress', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
for i, score in enumerate(tuning_scores, 1):
    plt.text(i, score + 0.005, f'{score:.4f}', ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig(f'/content/drive/MyDrive/AIvy - AI Powered Job Recommendation System/Results/Model 06 - outputs-05_tuning_progress.png', dpi=300, bbox_inches='tight')
plt.close()


 PART 2: HYPERPARAMETER TUNING (3 Iterations)

Iteration 1/3
Fitting 5 folds for each of 8 candidates, totalling 40 fits

Best Parameters: {'C': 10, 'max_iter': 1000, 'penalty': 'l2', 'solver': 'saga'}
Best CV Score: 0.8274
Time Taken: 160.38s

Iteration 2/3
Fitting 5 folds for each of 20 candidates, totalling 100 fits

Best Parameters: {'C': 5, 'max_iter': 1000, 'penalty': 'l2', 'solver': 'lbfgs'}
Best CV Score: 0.8234
Time Taken: 90.10s

Iteration 3/3
Fitting 5 folds for each of 10 candidates, totalling 50 fits

Best Parameters: {'C': 0.3, 'class_weight': None, 'max_iter': 2000, 'penalty': 'l2', 'solver': 'lbfgs'}
Best CV Score: 0.8188
Time Taken: 3.73s

Best Overall Parameters (Iteration 1):
{
    "C": 10,
    "max_iter": 1000,
    "penalty": "l2",
    "solver": "saga"
}


In [ ]:
# ============================================================================
# PART 3: CROSS-VALIDATION WITH BEST PARAMETERS
# ============================================================================

print("\nPART 3: CROSS-VALIDATION (5-Fold)")

final_model = LogisticRegression(**best_params, random_state=42, n_jobs=-1)

cv_scores = cross_val_score(
    final_model, X_train_scaled, y_train,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='f1_weighted',
    n_jobs=-1
)

print(f"\nCV Scores: {cv_scores}")
print(f"Mean CV Score: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

# Cross-validation visualization
plt.figure(figsize=(10, 6))
plt.plot(range(1, 6), cv_scores, marker='o', linestyle='-', linewidth=2, markersize=12, color='gold')
plt.axhline(y=cv_scores.mean(), color='r', linestyle='--', label=f'Mean: {cv_scores.mean():.4f}')
plt.fill_between(range(1, 6), cv_scores.mean() - cv_scores.std(),
                 cv_scores.mean() + cv_scores.std(), alpha=0.2, color='yellow')
plt.xlabel('Fold', fontsize=12, fontweight='bold')
plt.ylabel('F1 Score', fontsize=12, fontweight='bold')
plt.title('Logistic Regression Cross-Validation Scores', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'/content/drive/MyDrive/AIvy - AI Powered Job Recommendation System/Results/Model 06 - outputs-06_cv_scores.png', dpi=300, bbox_inches='tight')
plt.close()


PART 3: CROSS-VALIDATION (5-Fold)

CV Scores: [0.83075117 0.82708777 0.82380952 0.81279966 0.84230511]
Mean CV Score: 0.8274 (+/- 0.0096)


In [ ]:
# ============================================================================
# PART 4: FINAL MODEL TRAINING
# ============================================================================

print("\nPART 4: FINAL MODEL TRAINING")

start_time = time.time()
final_model.fit(X_train_scaled, y_train)
train_time = time.time() - start_time

print(f"Model trained in {train_time:.2f}s")
print(f"Number of iterations: {final_model.n_iter_}")

y_train_pred = final_model.predict(X_train_scaled)
y_test_pred = final_model.predict(X_test_scaled)
y_test_proba = final_model.predict_proba(X_test_scaled)

train_metrics = {
    'accuracy': accuracy_score(y_train, y_train_pred),
    'precision': precision_score(y_train, y_train_pred, average='weighted', zero_division=0),
    'recall': recall_score(y_train, y_train_pred, average='weighted', zero_division=0),
    'f1_score': f1_score(y_train, y_train_pred, average='weighted', zero_division=0)
}

test_metrics = {
    'accuracy': accuracy_score(y_test, y_test_pred),
    'precision': precision_score(y_test, y_test_pred, average='weighted', zero_division=0),
    'recall': recall_score(y_test, y_test_pred, average='weighted', zero_division=0),
    'f1_score': f1_score(y_test, y_test_pred, average='weighted', zero_division=0)
}

print("\nTRAINING SET METRICS:")
for metric, value in train_metrics.items():
    print(f"  {metric.capitalize()}: {value:.4f}")

print("\nTEST SET METRICS:")
for metric, value in test_metrics.items():
    print(f"  {metric.capitalize()}: {value:.4f}")


PART 4: FINAL MODEL TRAINING
Model trained in 6.56s
Number of iterations: [843]

TRAINING SET METRICS:
  Accuracy: 0.9938
  Precision: 0.9913
  Recall: 0.9938
  F1_score: 0.9919

TEST SET METRICS:
  Accuracy: 0.8600
  Precision: 0.8190
  Recall: 0.8600
  F1_score: 0.8390
